In [ ]:
import mne
import pandas as pd
import numpy as np
from pathlib import Path
import os
import random

random.seed(1)
np.random.seed(1)

# THINGS-EEG Data Loading

### Making it work for sub-01

In [2]:
# Path to your EEG data
base = Path(r"C:\Users\coolu\Documents\eeg_project\data\THINGS_EEG_1\sub-01\eeg")

# Load EEG data
vhdr_path = base / "sub-01_task-rsvp_eeg.vhdr"

In [ ]:
raw = mne.io.read_raw_brainvision(vhdr_path, preload=True)

# Load event metadata
events_tsv = base / "sub-01_task-rsvp_events.tsv"
events_df = pd.read_csv(events_tsv, sep='\t')

# Create event_id mapping (unique stimulus names → integer codes)
event_id = {name: idx + 1 for idx, name in enumerate(events_df["stimname"].unique())}

# Use the 'sample' column directly (already in samples)
onsets = events_df["sample"].astype(int).to_numpy()

# Create MNE events array (n_events x 3): [onset, 0, event_code]
events = np.column_stack((onsets, np.zeros_like(onsets), 
                          [event_id[name] for name in events_df["stimname"]]))

print(f"Loaded {len(events)} events, {len(event_id)} unique stimuli")

# Epoch the EEG around each stimulus
epochs = mne.Epochs(
    raw,
    events=events,
    event_id=event_id,
    tmin=-0.2,   # 200 ms pre-stimulus baseline
    tmax=0.8,    # 800 ms after stimulus onset
    baseline=(None, 0),
    preload=True,
    reject_by_annotation=False  # BrainVision files often contain segment markers
)
epochs.metadata = events_df
print(epochs)

Extracting parameters from C:\Users\coolu\Documents\eeg_project\data\THINGS_EEG_1\sub-01\eeg\sub-01_task-rsvp_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 3035739  =      0.000 ...  3035.739 secs...
Loaded 22248 events, 22248 unique stimuli
Not setting metadata
22248 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 22248 events and 1001 original time points ...
0 bad epochs dropped
<Epochs |  22248 events (all good), -0.2 – 0.8 s, baseline -0.2 – 0 s, ~10.45 GB, data loaded,
 'carousel_11s.jpg': 1
 'piano_05n.jpg': 1
 'turntable_12s.jpg': 1
 'tent_08n.jpg': 1
 'stove2_09s.jpg': 1
 'birdbath_08s.jpg': 1
 'drum_10n.jpg': 1
 'calf1_08s.jpg': 1
 'mole_05s.jpg': 1
 'stirrup_06s.jpg': 1
 and 22238 more events ...>


In [5]:
print(epochs.ch_names)
print(len(epochs.ch_names))

['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2', 'FCz']
63


In [ ]:
print(len(events_df), len(epochs))
if len(events_df) != len(epochs):
    print("oops")

22248 22248


In [ ]:
X = epochs.get_data()
stimnames = epochs.metadata["stimname"].values

print(f"EEG array shape: {X.shape}")  # (22248, 64, 1001)
print(f"Stimulus names: {len(stimnames)} entries")

eeg_trials = [X[i] for i in range(len(X))]

EEG array shape: (22248, 63, 1001)
Stimulus names: 22248 entries


C:\Users\coolu\AppData\Local\Temp\ipykernel_9672\3328076029.py:1: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X = epochs.get_data()


In [10]:
print(eeg_trials[0])

[[-2.53745789e-05 -2.27378615e-05 -1.51206779e-05 ...  3.62931251e-06
   4.36173401e-06 -6.67560293e-07]
 [-1.00947846e-05 -9.31353495e-06 -6.57916135e-06 ... -2.23346045e-06
  -7.19789355e-07 -1.45221085e-06]
 [-1.14857781e-05 -8.26312349e-06 -1.71498377e-05 ... -3.42914159e-06
  -2.98968869e-06 -2.84320439e-06]
 ...
 [-9.27855363e-06 -1.04504280e-05 -1.17687867e-05 ...  7.80034971e-07
  -2.54027583e-06 -2.63793203e-06]
 [-7.38348891e-06 -8.65301951e-06 -7.77411371e-06 ...  2.08916249e-06
  -2.01239791e-06 -1.86591361e-06]
 [-2.28641829e-06 -3.40946459e-06 -3.26298029e-06 ...  5.42842151e-06
   8.38580105e-07  2.52642905e-07]]


In [20]:
stimuli = []

for i in range(len(X)):
    stimname = epochs.metadata.loc[i, "stimname"]
    stim_base = stimname.split("_")[0]
    stim_time = epochs.metadata.loc[i, "stimdur"]

    entry = [
        X[i],  # eeg_data
        f"data/images_THINGS/object_images/{stimname}",  # image path
        stim_base,  # label
        stim_time,  # stim time
        1  # origin
    ]
    stimuli.append(entry)

print(stimuli[0])

[array([[-2.53745789e-05, -2.27378615e-05, -1.51206779e-05, ...,
         3.62931251e-06,  4.36173401e-06, -6.67560293e-07],
       [-1.00947846e-05, -9.31353495e-06, -6.57916135e-06, ...,
        -2.23346045e-06, -7.19789355e-07, -1.45221085e-06],
       [-1.14857781e-05, -8.26312349e-06, -1.71498377e-05, ...,
        -3.42914159e-06, -2.98968869e-06, -2.84320439e-06],
       ...,
       [-9.27855363e-06, -1.04504280e-05, -1.17687867e-05, ...,
         7.80034971e-07, -2.54027583e-06, -2.63793203e-06],
       [-7.38348891e-06, -8.65301951e-06, -7.77411371e-06, ...,
         2.08916249e-06, -2.01239791e-06, -1.86591361e-06],
       [-2.28641829e-06, -3.40946459e-06, -3.26298029e-06, ...,
         5.42842151e-06,  8.38580105e-07,  2.52642905e-07]]), 'data/images_THINGS/object_images/carousel_11s.jpg', 'carousel', 0.0499804588034749, 1]


In [ ]:
save_path = os.path.join("..", "data", "reformatted", "sub-01_unprocessed.npy")

# Ensure the directory exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save the file
np.save(save_path, np.array(stimuli, dtype=object))

print(f"Saved {len(stimuli)} entries to {save_path}")

Saved 22248 entries to ..\data\reformatted\sub-01_unprocessed.npy


### Iterating through all subjects

In [ ]:
first_sub = 1
last_sub = 2
for sub in range(first_sub, last_sub + 1):
    sub_name = f"sub-{sub:02}"  # e.g. sub-01, sub-10
    print("Starting on " + sub_name)
    base = Path(r"C:\Users\coolu\Documents\eeg_project\data\THINGS_EEG_1") / sub_name / "eeg"
    vhdr_path = base / f"{sub_name}_task-rsvp_eeg.vhdr"
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=True)

    # Load event metadata
    events_path = sub_name + "_task-rsvp_events.tsv"
    events_tsv = base / events_path
    events_df = pd.read_csv(events_tsv, sep='\t')

    # Create event_id mapping (unique stimulus names → integer codes)
    event_id = {name: idx + 1 for idx, name in enumerate(events_df["stimname"].unique())}

    # Use the 'sample' column directly (already in samples)
    onsets = events_df["sample"].astype(int).to_numpy()

    # Create MNE events array (n_events x 3): [onset, 0, event_code]
    events = np.column_stack((onsets, np.zeros_like(onsets), 
                            [event_id[name] for name in events_df["stimname"]]))

    print(f"Loaded {len(events)} events, {len(event_id)} unique stimuli for {sub_name}")

    # Epoch the EEG around each stimulus
    epochs = mne.Epochs(
        raw,
        events=events,
        event_id=event_id,
        tmin=-0.2,   # 200 ms pre-stimulus baseline
        tmax=0.8,    # 800 ms after stimulus onset
        baseline=(None, 0),
        preload=True,
        reject_by_annotation=False  # BrainVision files often contain segment markers
    )
    epochs.metadata = events_df
    print(epochs)
    
    print(len(events_df), len(epochs))
    if len(events_df) != len(epochs):
        print("oops for" + sub_name)
        
    X = epochs.get_data()
    stimnames = epochs.metadata["stimname"].values

    print(f"EEG array shape: {X.shape}")  # (22248, 64, 1001)
    print(f"Stimulus names: {len(stimnames)} entries")

    eeg_trials = [X[i] for i in range(len(X))]
    
    stimuli = []

    for i in range(len(X)):
        stimname = epochs.metadata.loc[i, "stimname"]
        stim_base = stimname.split("_")[0]
        stim_time = epochs.metadata.loc[i, "stimdur"]

        entry = [
            X[i],  # eeg_data
            f"data/images_THINGS/object_images/{stim_base + '/' +stimname}",  # image path
            stim_base,  # label
            stim_time,  # stim time
            1  # origin
        ]
        stimuli.append(entry)

    print(stimuli[0])
    
    print(f"Saving {len(stimuli)} entries to {save_path} for {sub_name}")
    save_path = os.path.join("..", "data", "reformatted", f"{sub_name}_unprocessed.npy")

    # Ensure the directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Save the file
    np.save(save_path, np.array(stimuli, dtype=object))

    print(f"Saved {len(stimuli)} entries to {save_path} for {sub_name}")
    print()

Starting on sub-01
Extracting parameters from C:\Users\coolu\Documents\eeg_project\data\THINGS_EEG_1\sub-01\eeg\sub-01_task-rsvp_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 3035739  =      0.000 ...  3035.739 secs...
Loaded 22248 events, 22248 unique stimuli for sub-01
Not setting metadata
22248 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 22248 events and 1001 original time points ...
0 bad epochs dropped
Adding metadata with 20 columns
<Epochs |  22248 events (all good), -0.2 – 0.8 s, baseline -0.2 – 0 s, ~10.45 GB, data loaded, with metadata,
 'carousel_11s.jpg': 1
 'piano_05n.jpg': 1
 'turntable_12s.jpg': 1
 'tent_08n.jpg': 1
 'stove2_09s.jpg': 1
 'birdbath_08s.jpg': 1
 'drum_10n.jpg': 1
 'calf1_08s.jpg': 1
 'mole_05s.jpg': 1
 'stirrup_06s.jpg': 1
 and 22238 more events ...>
22248 22248
EEG array shape: (22248, 63, 1001)
Stimulus names: 22248 

C:\Users\coolu\AppData\Local\Temp\ipykernel_12284\562235122.py:45: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X = epochs.get_data()


[array([[-2.53745789e-05, -2.27378615e-05, -1.51206779e-05, ...,
         3.62931251e-06,  4.36173401e-06, -6.67560293e-07],
       [-1.00947846e-05, -9.31353495e-06, -6.57916135e-06, ...,
        -2.23346045e-06, -7.19789355e-07, -1.45221085e-06],
       [-1.14857781e-05, -8.26312349e-06, -1.71498377e-05, ...,
        -3.42914159e-06, -2.98968869e-06, -2.84320439e-06],
       ...,
       [-9.27855363e-06, -1.04504280e-05, -1.17687867e-05, ...,
         7.80034971e-07, -2.54027583e-06, -2.63793203e-06],
       [-7.38348891e-06, -8.65301951e-06, -7.77411371e-06, ...,
         2.08916249e-06, -2.01239791e-06, -1.86591361e-06],
       [-2.28641829e-06, -3.40946459e-06, -3.26298029e-06, ...,
         5.42842151e-06,  8.38580105e-07,  2.52642905e-07]]), 'data/images_THINGS/object_images/carousel_11s.jpg', 'carousel', 0.0499804588034749, 1]
Saved 22248 entries to ..\data\reformatted\sub-01-_unprocessed.npy for sub-01

Starting on sub-02
Extracting parameters from C:\Users\coolu\Documents\eeg

C:\Users\coolu\AppData\Local\Temp\ipykernel_12284\562235122.py:45: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X = epochs.get_data()


[array([[ 1.16509434e-04,  1.06450845e-04,  1.00884442e-04, ...,
        -3.27289167e-04, -3.13177846e-04, -3.02875117e-04],
       [ 6.70764485e-05,  5.30139557e-05,  3.76819323e-05, ...,
        -8.65367541e-05, -7.69664465e-05, -6.93492629e-05],
       [ 3.54108183e-05,  3.11627736e-05,  3.18951951e-05, ...,
        -3.79289879e-05, -2.66008687e-05, -1.93743099e-05],
       ...,
       [ 1.23962643e-05,  1.09314213e-05,  1.36169668e-05, ...,
        -1.55334089e-05, -8.25802205e-06, -4.98653935e-06],
       [ 1.51925840e-05,  1.59250055e-05,  2.32003924e-05, ...,
        -6.21022983e-05, -3.92995756e-05, -1.11257619e-05],
       [ 1.15392947e-04,  1.14562869e-04,  1.17492555e-04, ...,
        -2.58874440e-04, -2.43884213e-04, -2.34362734e-04]]), 'data/images_THINGS/object_images/volleyball_08s.jpg', 'volleyball', 0.0499782034603413, 1]
Saved 24648 entries to ..\data\reformatted\sub-02-_unprocessed.npy for sub-02



### Checking

In [ ]:
import numpy as np
import pandas as pd

data = np.load("../data/reformatted/sub-01_unprocessed.npy", allow_pickle=True)

print(f"Total trials: {len(data)}")

Total trials: 22248
0: label=carousel, stim_time=0.0499804588034749, eeg_shape=(63, 1001), img_path=data/images_THINGS/object_images/carousel
1: label=piano, stim_time=0.0499772892799228, eeg_shape=(63, 1001), img_path=data/images_THINGS/object_images/piano
2: label=turntable, stim_time=0.049983628327027, eeg_shape=(63, 1001), img_path=data/images_THINGS/object_images/turntable
3: label=tent, stim_time=0.0499514453113079, eeg_shape=(63, 1001), img_path=data/images_THINGS/object_images/tent
4: label=stove2, stim_time=0.0499738759826869, eeg_shape=(63, 1001), img_path=data/images_THINGS/object_images/stove2


In [18]:
import pandas as pd

sub_name = "sub-01"
base = Path(r"C:\Users\coolu\Documents\eeg_project\data\THINGS_EEG_1") / sub_name / "eeg"
events_path = sub_name + "_task-rsvp_events.tsv"
events_tsv = base / events_path
meta = pd.read_csv(events_tsv, sep='\t')
stimnames = meta["stimname"].values

# Compare labels from your saved data
labels_from_npy = [d[2] for d in data]  # index 2 = label

# Check that the unique counts and order roughly align
print("Unique stimuli in metadata:", len(set(stimnames)))
print("Unique labels in EEG data:", len(set(labels_from_npy)))

# Optionally verify order matches exactly for a sample
for i in range(5):
    print(stimnames[i], "↔", labels_from_npy[i])


Unique stimuli in metadata: 22248
Unique labels in EEG data: 1752
carousel_11s.jpg ↔ carousel
piano_05n.jpg ↔ piano
turntable_12s.jpg ↔ turntable
tent_08n.jpg ↔ tent
stove2_09s.jpg ↔ stove2


# THINGS-EEG2 Data Loading

In [38]:
# cycle through all 16540 images
# cycle through each of the 4 reps
# save 17x100 array with image label
data_path = "../data/THINGS_EEG_2"
num_subjects = 1
stimdur = .10
origin = 2

train_data_matrix = []
valid_data_matrix = []
test_data_matrix = []

img_metadata = np.load(os.path.join(data_path, 'image_metadata.npy'), allow_pickle=True).item()

for sub in range(num_subjects):
    subject = sub+1
    print("Loading subject " + str(subject) + '...')
    eeg_data_train = np.load(os.path.join(data_path, 'sub-'+str(subject), 'preprocessed_eeg_training.npy'), allow_pickle=True).item()
    
    train_data_dim = eeg_data_train['preprocessed_eeg_data'].shape
    images = 16 #data_dim[0]
    repitition = train_data_dim[1]
    channels = train_data_dim[2]
    times = train_data_dim[3]
    
    data_matrix = []
    label_matrix = []
    for im in range(images):
        full_label = img_metadata['train_img_concepts'][im] + '_' + img_metadata['train_img_files'][im]
        print(full_label)
        image_path = "_".join(full_label.split("_")[2:])
        print(image_path)
        label = full_label.split("_")[1]
        print(label)
        #print(image_label)
        for rep in range(repitition):
            trial_data = eeg_data_train['preprocessed_eeg_data'][im][rep]
            entry = [
                trial_data,  # eeg_data
                f"data/images_THINGS/object_images/{label + '/' + image_path}",  # image path
                label,  # label
                stimdur,  # stim time
                origin  # origin
            ]
            data_matrix.append(entry)

    data_matrix = np.array(data_matrix)
    print(data_matrix)
    print(data_matrix.shape)
    print("Done with subject " + str(subject))

#train_data_matrix = np.array(train_data_matrix)
#print(train_data_matrix.shape)

'''
valid_data_matrix = np.array(valid_data_matrix)
print(valid_data_matrix.shape)

test_data_matrix = np.array(test_data_matrix)
print(test_data_matrix.shape)
'''

Loading subject 1...
00001_aardvark_aardvark_01b.jpg
aardvark_01b.jpg
aardvark
00001_aardvark_aardvark_02s.jpg
aardvark_02s.jpg
aardvark
00001_aardvark_aardvark_03s.jpg
aardvark_03s.jpg
aardvark
00001_aardvark_aardvark_04s.jpg
aardvark_04s.jpg
aardvark
00001_aardvark_aardvark_05s.jpg
aardvark_05s.jpg
aardvark
00001_aardvark_aardvark_06s.jpg
aardvark_06s.jpg
aardvark
00001_aardvark_aardvark_07s.jpg
aardvark_07s.jpg
aardvark
00001_aardvark_aardvark_08s.jpg
aardvark_08s.jpg
aardvark
00001_aardvark_aardvark_09s.jpg
aardvark_09s.jpg
aardvark
00001_aardvark_aardvark_10s.jpg
aardvark_10s.jpg
aardvark
00002_abacus_abacus_01b.jpg
abacus_01b.jpg
abacus
00002_abacus_abacus_02s.jpg
abacus_02s.jpg
abacus
00002_abacus_abacus_03s.jpg
abacus_03s.jpg
abacus
00002_abacus_abacus_04s.jpg
abacus_04s.jpg
abacus
00002_abacus_abacus_05s.jpg
abacus_05s.jpg
abacus
00002_abacus_abacus_06s.jpg
abacus_06s.jpg
abacus
[[array([[-0.80418691, -0.39927395,  0.07786927, ..., -0.84295037,
          -0.3065356 ,  0.035217

C:\Users\coolu\AppData\Local\Temp\ipykernel_12284\165183188.py:47: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  data_matrix = np.array(data_matrix)


'\nvalid_data_matrix = np.array(valid_data_matrix)\nprint(valid_data_matrix.shape)\n\ntest_data_matrix = np.array(test_data_matrix)\nprint(test_data_matrix.shape)\n'

## Old Code, possibly useful, possibly not

In [ ]:
i = 0
for im in obj_type:
    image_label = int(np.floor(i/10))
    split_choice = random.randint(0, 3)
    for rep in range(repitition):
        if rep == split_choice: # take one repitition of each image from each subject
            trial_data = eeg_data_train['preprocessed_eeg_data'][im][rep]
            if n%4==0 or n%5==0:
                test_data_matrix.append(trial_data)
                test_label_matrix.append(image_label)
            else:
                valid_data_matrix.append(trial_data)
                valid_label_matrix.append(image_label)
            n+=1
        else:
            trial_data = eeg_data_train['preprocessed_eeg_data'][im][rep]
            train_data_matrix.append(trial_data)
            train_label_matrix.append(image_label)
    i += 1

In [ ]:
indices = random.sample(range(len(train_data_matrix)), len(train_data_matrix))  # Get a shuffled list of indices
train_data_matrix = [train_data_matrix[i] for i in indices]  

indices = random.sample(range(len(valid_data_matrix)), len(valid_data_matrix))  # Get a shuffled list of indices
valid_data_matrix = [valid_data_matrix[i] for i in indices]  

indices = random.sample(range(len(test_data_matrix)), len(test_data_matrix))  # Get a shuffled list of indices
test_data_matrix = [test_data_matrix[i] for i in indices]  

# Convert the data and labels to NumPy arrays
train_data_matrix = np.array(train_data_matrix)
valid_data_matrix = np.array(valid_data_matrix)
test_data_matrix = np.array(test_data_matrix)

# Normalize data
train_data_matrix = train_data_matrix / np.linalg.norm(train_data_matrix, axis=(1,2), keepdims=True)
valid_data_matrix = valid_data_matrix / np.linalg.norm(valid_data_matrix, axis=(1,2), keepdims=True)
test_data_matrix = test_data_matrix / np.linalg.norm(test_data_matrix, axis=(1,2), keepdims=True)

# Reshape the input data to have the expected shape for the CNN
train_data_matrix = train_data_matrix.reshape(-1, 17, 100, 1)
valid_data_matrix = valid_data_matrix.reshape(-1, 17, 100, 1)
test_data_matrix = test_data_matrix.reshape(-1, 17, 100, 1)

# Ensure the input data is of type float32
train_data_matrix = train_data_matrix.astype('float32')
valid_data_matrix = valid_data_matrix.astype('float32')
test_data_matrix = test_data_matrix.astype('float32')